## 1) Load Data

Load data, filter by removing unnecessary columns(data), and process the rest of it

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.signal import find_peaks

In [14]:
BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name != "deeplab-insect-locomotion-analysis":
    for parent in BASE_DIR.parents:
        if parent.name == "deeplab-insect-locomotion-analysis":
            BASE_DIR = parent
            print(f"Base directory found: {parent}")
            break

# Setting the current working directory to the base directory of the project (pwd so even Path.cwd() will return the base directory)
os.chdir(BASE_DIR)

from helper import parse_file_trial_duration_info, filter_dataframe

In [ ]:
# Reset to default white theme and explicitly set face colors instead of using system theme colors
plt.style.use("default")
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["savefig.facecolor"] = "white"

In [ ]:
# FILE_DIR = str(Path('path')) # Replace 'path' with the actual path to your directory

FILES_DIR = [str(path) for path in Path("videos_jetson").glob("*.csv")]
FILES_DIR.sort()
FILES_PROTRACTOR_DIR = [str(path) for path in Path("videos_jetson_protractor").glob("*.csv")]
FILES_PROTRACTOR_DIR.sort()

FILES_EX11_DIR = [
    str(path)
    for path in Path("videos_jetson_experiment_11").glob(
        "*sDLC_Resnet50_Stick_Insect_Motion_Hacking_2D_TrackingOct2shuffle2_snapshot_1000.csv"
    )
]
FILES_EX11_DIR.sort()
experiment_11_files = FILES_EX11_DIR
experiment_11_files_ind03 = list(filter(lambda file: parse_file_trial_duration_info(file)["ind_num"] == 3, FILES_EX11_DIR))
experiment_11_files_ind04 = list(filter(lambda file: parse_file_trial_duration_info(file)["ind_num"] == 4, FILES_EX11_DIR))

FILES_EX12_DIR = [str(path) for path in Path("videos_jetson_experiment_12").glob("*.csv")]
FILES_EX12_DIR.sort()
experiment_12_files_ind05 = list(filter(lambda file: parse_file_trial_duration_info(file)["ind_num"] == 5, FILES_EX12_DIR))
experiment_12_files_ind06 = list(filter(lambda file: parse_file_trial_duration_info(file)["ind_num"] == 6, FILES_EX12_DIR))

FILES_EX13_DIR = [str(path) for path in Path("videos_jetson_experiment_13").glob("*.csv")]
FILES_EX13_DIR.sort()

FILES_EX14_DIR = [str(path) for path in Path("videos_jetson_experiment_14").glob("*.csv")]
FILES_EX14_DIR.sort()

FILES_EX15_DIR = [str(path) for path in Path("videos_jetson_experiment_15").glob("*.csv")]
FILES_EX15_DIR.sort()

## 2) Mapping of Plot details

Create maps for legend details for specific experiment parameters and specify their adequate color

In [ ]:
# Map results, give a legend color,
trial_map = {
    0: {"trial_detail": "Others", "color": "#a734d5"},
    1: {"trial_detail": "Random (No Stimulation)", "color": "#f77189"},
    2: {"trial_detail": "Random (No Stimulation)", "color": "#f77189"},
    3: {"trial_detail": "Random (No Stimulation)", "color": "#f77189"},
    4: {"trial_detail": "Random (No Stimulation)", "color": "#f77189"},
    5: {"trial_detail": "0.1s with 1.0s period (T)", "color": "#e68332"},
    6: {"trial_detail": "0.1s with 1.0s period (T)", "color": "#e68332"},
    7: {"trial_detail": "0.5s with 1.5s period (T)", "color": "#bb9832"},
    8: {"trial_detail": "0.5s with 1.5s period (T)", "color": "#bb9832"},
    9: {"trial_detail": "1.0s with 2.0s period (T)", "color": "#97a431"},
    10: {"trial_detail": "1.0s with 2.0s period (T)", "color": "#97a431"},
    11: {"trial_detail": "1.5s with 2.5s period (T)", "color": "#50b131"},
    12: {"trial_detail": "1.5s with 2.5s period (T)", "color": "#50b131"},
    13: {"trial_detail": "2.0s with 3.0s period (T)", "color": "#34af84"},
    14: {"trial_detail": "2.0s with 3.0s period (T)", "color": "#34af84"},
    15: {"trial_detail": "2.5s with 3.5s period (T)", "color": "#36ada4"},
    16: {"trial_detail": "2.5s with 3.5s period (T)", "color": "#36ada4"},
    17: {"trial_detail": "3.0s with 4.0s period (T)", "color": "#38aabf"},
    18: {"trial_detail": "3.0s with 4.0s period (T)", "color": "#38aabf"},
    19: {"trial_detail": "3.5s with 4.5s period (T)", "color": "#3ba3ec"},
    20: {"trial_detail": "3.5s with 4.5s period (T)", "color": "#3ba3ec"},
    21: {"trial_detail": "4.0s with 5.0s period (T)", "color": "#a48cf4"},
    22: {"trial_detail": "4.0s with 5.0s period (T)", "color": "#a48cf4"},
    23: {"trial_detail": "4.5s with 5.5s period (T)", "color": "#e866f4"},
    24: {"trial_detail": "4.5s with 5.5s period (T)", "color": "#e866f4"},
    25: {"trial_detail": "5.0s with 6.0s period (T)", "color": "#f668c2"},
    26: {"trial_detail": "5.0s with 6.0s period (T)", "color": "#f668c2"},
    27: {"trial_detail": "0.2s with 1.5s period (T)", "color": "#110539"},
    28: {"trial_detail": "0.2s with 1.5s period (T)", "color": "#110539"},
    29: {"trial_detail": "0.3s with 1.5s period (T)", "color": "#850913"},
    30: {"trial_detail": "0.3s with 1.5s period (T)", "color": "#850913"},
    31: {"trial_detail": "0.4s with 1.5s period (T)", "color": "#edfa58"},
    32: {"trial_detail": "0.4s with 1.5s period (T)", "color": "#edfa58"},
    33: {"trial_detail": "0.6s with 1.5s period (T)", "color": "#cd4948"},
    34: {"trial_detail": "0.6s with 1.5s period (T)", "color": "#cd4948"},
    35: {"trial_detail": "0.7s with 1.5s period (T)", "color": "#58ea70"},
    36: {"trial_detail": "0.7s with 1.5s period (T)", "color": "#58ea70"},
    37: {"trial_detail": "0.8s with 1.5s period (T)", "color": "#7f963f"},
    38: {"trial_detail": "0.8s with 1.5s period (T)", "color": "#7f963f"},
    39: {"trial_detail": "0.9s with 1.5s period (T)", "color": "#c9362e"},
    40: {"trial_detail": "0.9s with 1.5s period (T)", "color": "#c9362e"},
    41: {"trial_detail": "Manual Stimulation", "color": "#1acbc5"},
    42: {"trial_detail": "Manual Stimulation", "color": "#1acbc5"},
    43: {"trial_detail": "0.5s with 1s period (T), 10% Duty Ratio", "color": "#8d88db"},
    44: {"trial_detail": "0.5s with 1s period (T), 20% Duty Ratio", "color": "#222283"},
    45: {"trial_detail": "0.5s with 1s period (T), 30% Duty Ratio", "color": "#ec71e1"},
    46: {"trial_detail": "0.5s with 1s period (T), 40% Duty Ratio", "color": "#0c9a4b"},
    47: {"trial_detail": "0.5s with 1s period (T), 50% Duty Ratio", "color": "#a4edc5"},
    48: {"trial_detail": "0.5s with 1.5s period (T), 1280x720 Resolution", "color": "#ede3a4"},
    49: {"trial_detail": "1.0s with 2.0s period (T), 1280x720 Resolution", "color": "#aaa4ed"},
    50: {"trial_detail": "2.0s with 3.0s period (T), 1280x720 Resolution", "color": "#eda4e0"},
    51: {"trial_detail": "2.0s with 3.0s period (T), 1280x720 Resolution", "color": "#875039"},
    110: {"trial_detail": "Auto Exposure", "color": "#e84d4d"},
    120: {"trial_detail": "Manual Exposure (Exposure 70)", "color": "#4b1904"},
    130: {"trial_detail": "Manual Exposure (Exposure 70, Light 20)", "color": "#066309"},
    140: {"trial_detail": "Manual Exposure (Exposure 70, Light 50)", "color": "#25b3b8"},
}


In [ ]:
trial_protractor_map = {
    "0.0": {"trial_detail": "No Stimulation", "color": "#a734d5"},
    "0.1": {"trial_detail": "0.1s with 1.0s period (T)", "color": "#e68332"},
    "0.5": {"trial_detail": "0.5s with 1.5s period (T)", "color": "#bb9832"},
    "1.0": {"trial_detail": "1.0s with 2.0s period (T)", "color": "#97a431"},
    "1.5": {"trial_detail": "1.5s with 2.5s period (T)", "color": "#50b131"},
    "2.0": {"trial_detail": "2.0s with 3.0s period (T)", "color": "#34af84"},
    "2.5": {"trial_detail": "2.5s with 3.5s period (T)", "color": "#36ada4"},
    "3.0": {"trial_detail": "3.0s with 4.0s period (T)", "color": "#38aabf"},
    "3.5": {"trial_detail": "3.5s with 4.5s period (T)", "color": "#3ba3ec"},
    "4.0": {"trial_detail": "4.0s with 5.0s period (T)", "color": "#a48cf4"},
    "4.5": {"trial_detail": "4.5s with 5.5s period (T)", "color": "#e866f4"},
    "5.0": {"trial_detail": "5.0s with 6.0s period (T)", "color": "#f668c2"},
}

In [ ]:
trial_freq_map = {
    "10": {"trial_detail": "10Hz", "color": "#a734d5"},
    "20": {"trial_detail": "20Hz", "color": "#7341e8"},
    "30": {"trial_detail": "30Hz", "color": "#3b62f6"},
    "40": {"trial_detail": "40Hz", "color": "#2a9df4"},
    "50": {"trial_detail": "50Hz", "color": "#e68332"},
    "60": {"trial_detail": "60Hz", "color": "#e5b832"},
    "70": {"trial_detail": "70Hz", "color": "#b1e532"},
    "80": {"trial_detail": "80Hz", "color": "#5fe532"},
    "90": {"trial_detail": "90Hz", "color": "#2b6802"},
    "100": {"trial_detail": "100Hz", "color": "#36ada4"},
    "110": {"trial_detail": "110Hz", "color": "#32a9e5"},
    "120": {"trial_detail": "120Hz", "color": "#326ee5"},
    "130": {"trial_detail": "130Hz", "color": "#6832e5"},
    "150": {"trial_detail": "150Hz", "color": "#97a431"},
    "200": {"trial_detail": "200Hz", "color": "#e866f4"},
    "250": {"trial_detail": "250Hz", "color": "#34af84"},
    "300": {"trial_detail": "300Hz", "color": "#f4426b"},
    "350": {"trial_detail": "350Hz", "color": "#f49142"},
    "400": {"trial_detail": "400Hz", "color": "#d82bff"},
    "450": {"trial_detail": "450Hz", "color": "#42f4c5"},
    "500": {"trial_detail": "500Hz", "color": "#210d44"},
    "550": {"trial_detail": "550Hz", "color": "#191cf0"},
    "600": {"trial_detail": "600Hz", "color": "#9cf442"},
    "650": {"trial_detail": "650Hz", "color": "#42d4f4"},
}

## 3) Plotting Data For Each Experiment

In [ ]:
output_folder = Path("trial_plots")
output_folder.mkdir(parents=True, exist_ok=True)

for file in FILES_DIR:
    df = pd.read_csv(file)

    special_overrides = {
        "011": "110",  # Mapped formatted string
        "012": "120",
        "013": "130",
        "014": "140",
    }

    # Extract trial label (e.g. '01', '02', '14', etc.)
    match = re.search(r"ind(?:_)?01_(\d+)", file)
    if match:
        raw_id = match.group(1)  # e.g., '07' or '071'

        # Insert decimal point if ID has 3 or more digits ('071' -> '07.1')
        if raw_id in ("011", "012", "013", "014"):
            formatted_id = special_overrides[raw_id]
        elif len(raw_id) >= 3:
            formatted_id = f"{raw_id[:-1]}.{raw_id[-1]}"
        else:
            formatted_id = raw_id

        trial_num = float(formatted_id)  # 7.1 (allows range check 7 <= 7.1 <= 26)
        base_trial = int(trial_num)  # 7 (used to lookup key in trial_map)
        trial_label = f"Trial {formatted_id}"  # "Trial 07.1"
    else:
        trial_num = None
        base_trial = None
        trial_label = "Test"

    df_filtered = filter_dataframe(df)

    # Plot angle over time
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(28, 16), layout="constrained", gridspec_kw={"height_ratios": [1, 2, 1, 1]})

    # Title for entire figure
    fig.suptitle(
        f"Trial {trial_num} - {trial_map[int(trial_num)]['trial_detail'] if trial_num is not None else 'Test'}",
        fontsize=16,
        fontweight="bold",
    )

    # Coordinates over time
    ax1.plot(df_filtered["time"], df_filtered["TC_Middle_Leg_x"], label="TC_Middle_Leg X", color="red", alpha=0.7)
    ax1.plot(df_filtered["time"], df_filtered["TC_Middle_Leg_y"], label="TC_Middle_Leg Y", color="blue", alpha=0.7)
    ax1.plot(df_filtered["time"], df_filtered["FT_Middle_Leg_x"], label="FT_Middle_Leg X", color="darkred", linestyle="--")
    ax1.plot(df_filtered["time"], df_filtered["FT_Middle_Leg_y"], label="FT_Middle_Leg Y", color="darkblue", linestyle="--")

    ax1.set_title("Coordinates of TC and FT Middle Legs Over Time")
    ax1.set_ylabel("Coordinates")
    ax1.set_xlabel("Time (s)")
    ax1.legend()

    # Spatial trajectory of keypoints
    ax2.plot(df_filtered["TC_Front_Leg_x"], df_filtered["TC_Front_Leg_y"], color="blue")
    ax2.plot(df_filtered["TC_Middle_Leg_x"], df_filtered["TC_Middle_Leg_y"], color="green")
    ax2.plot(df_filtered["FT_Middle_Leg_x"], df_filtered["FT_Middle_Leg_y"], color="red")

    ax2.set_title("Spatial Trajectory of Keypoints")
    ax2.set_xlabel("X Coordinate")
    ax2.set_ylabel("Y Coordinate")
    ax2.invert_yaxis()  # Invert y-axis to match image coordinate system
    ax2.set_aspect("equal", adjustable="box")
    ax2.grid(True, alpha=0.3)

    # Angle over time
    ax3.plot(df_filtered["time"], df_filtered["angle_from_initial"], color="purple")
    ax3.set_title("Angle Between TCTC and TCFT Vectors Over Time")
    ax3.set_xlabel("Time (s)")
    ax3.set_ylabel("Angle (degrees)")

    ax3.set_xlim(0, 20)  # Set x-axis limit for time from 0 to 20 seconds

    # Angle difference
    # Collect peak differences and labels for each group
    angle_signal = df_filtered["angle_from_initial"].dropna().values
    peaks, properties = find_peaks(angle_signal, prominence=1)
    peak_prominences = properties["prominences"]

    if len(peak_prominences) > 0:
        # horizontal Violin Plot (vert=False) placed at y = 1
        parts = ax4.violinplot([peak_prominences], positions=[1], vert=False, showmeans=True, showmedians=False)

        # Custom styling
        for pc in parts["bodies"]:
            pc.set_facecolor("purple")
            pc.set_edgecolor("black")
            pc.set_alpha(0.3)

        # add vertical jitter around y = 1 (X stays exact, Y varies slightly)
        y_coords = np.random.normal(loc=1.0, scale=0.04, size=len(peak_prominences))
        ax4.scatter(peak_prominences, y_coords, alpha=0.7, s=50, color="darkblue", edgecolors="w", zorder=3)

    # 4. Axes Formatting
    ax4.set_yticks([1])
    ax4.set_yticklabels([f"{trial_label}"])
    ax4.set_ylim(0.5, 1.5)  # Keeps the trial centered along the Y-axis

    ax4.set_xlabel("Peak Amplitude / Difference (degrees)", fontsize=12)
    ax4.set_ylabel("Trial", fontsize=12)
    ax4.set_title(
        f"Peak Amplitude Distribution - {trial_label} ({trial_map[int(trial_num)]['trial_detail'] if trial_num is not None else 'Test'})",
        fontsize=14,
        pad=15,
    )
    ax4.grid(True, axis="x", linestyle=":", alpha=0.6)

    plt.subplots_adjust(hspace=0.6)
    # Save figure
    fig_path = output_folder / f"plot_wave_{trial_label}.png"
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.tight_layout()

In [ ]:
records = []

for file in FILES_DIR:
    df = pd.read_csv(file)

    special_overrides = {
        "011": "110",  # Mapped formatted string
        "012": "120",
        "013": "130",
        "014": "140",
    }

    # Extract trial label (e.g. '01', '02', '14', etc.)
    match = re.search(r"ind(?:_)?01_(\d+)", file)
    if match:
        raw_id = match.group(1)  # e.g., '07' or '071'

        # Insert decimal point if ID has 3 or more digits ('071' -> '07.1')
        if raw_id in ("011", "012", "013", "014"):
            formatted_id = special_overrides[raw_id]
        elif len(raw_id) >= 3:
            formatted_id = f"{raw_id[:-1]}.{raw_id[-1]}"
        else:
            formatted_id = raw_id

        trial_num = float(formatted_id)  # 7.1 (allows range check 7 <= 7.1 <= 26)
        base_trial = int(trial_num)  # 7 (used to lookup key in trial_map)
        trial_label = f"Trial {formatted_id}"  # "Trial 07.1"
    else:
        trial_num = None
        base_trial = None
        trial_label = "Test"

    df_filtered = filter_dataframe(df)

    # Find angle difference
    angle_signal = df_filtered["angle"].dropna().values
    peaks, properties = find_peaks(angle_signal, prominence=1)
    prominences = properties["prominences"]

    for p in prominences:
        if trial_num is None:
            continue

        records.append(
            {
                "Trial_Num": trial_num,
                "Trial": trial_label,
                "Peak_Difference": p,
                "color": trial_map[int(trial_num)]["color"],
                "trial_legend": trial_map[int(trial_num)]["trial_detail"],
            }
        )

summary_df = pd.DataFrame(records)
color_palette = dict(zip(summary_df["trial_legend"], summary_df["color"]))


# Plot angle over time
fig, (ax1, ax2, ax3, ax4, ax5, ax6, ax7) = plt.subplots(
    7,
    1,
    figsize=(18, 22),
    layout="constrained",
    gridspec_kw={"height_ratios": [2, 1, 1, 1, 1, 1, 1]},
)

# Add new column to select the modified trial names
if "Base_Key" not in summary_df.columns:
    summary_df["Base_Key"] = summary_df["Trial_Num"].apply(lambda x: int(x))

subplot_configs = [
    ("1. All Trials", None, None, ax1, (0, 180)),
    ("2. Trials 5 to 26 - Stimulation Duration 0.1 to 5.0s (with 0.5 increment)", 5, 26, ax2, (0, 20)),
    ("3. Trials 27 to 40 - Stimulation Duration 0.2 to 0.9s (with 0.1 increment)", 27, 40, ax3, (0, 25)),
    ("4. Trials 41 to 42", 41, 42, ax4, (0, 20)),
    ("5. Trials 43 to 47", 43, 47, ax5, (0, 12)),
    ("6. Trials 48 to 51", 48, 51, ax6, (0, 180)),
    ("7. Trials 110 to 140", 110, 140, ax7, (0, 180)),
]

for title, min_k, max_k, ax, ylim in subplot_configs:
    # Filter dataframe by range
    if min_k is None and max_k is None:
        sub_df = summary_df.copy()
    else:
        sub_df = summary_df[(summary_df["Base_Key"] >= min_k) & (summary_df["Base_Key"] <= max_k)]

    if sub_df.empty:
        ax.set_title(f"{title} (No Data Available)", fontsize=12)
        continue

    # Boxplot
    sns.boxplot(
        data=sub_df,
        x="Trial",
        y="Peak_Difference",
        hue="trial_legend",
        palette=color_palette,
        dodge=False,
        boxprops=dict(alpha=0.4),
        showfliers=False,
        legend=False,  # Suppress subplot legend to prevent clutter
        ax=ax,
    )

    # Stripplot
    sns.stripplot(
        data=sub_df,
        x="Trial",
        y="Peak_Difference",
        hue="trial_legend",
        palette=color_palette,
        dodge=False,
        size=5,
        jitter=0.2,
        alpha=0.8,
        legend=False,
        ax=ax,
    )

    if ax != ax1:
        sub_palette = dict(zip(sub_df["trial_legend"], sub_df["color"]))
        sub_patches = [mpatches.Patch(color=c, label=l) for l, c in sub_palette.items()]

        ax.legend(
            handles=sub_patches,
            title="Stimulus Detail",
            bbox_to_anchor=(1.02, 1),  # Places legend just outside the right edge of the subplot
            loc="upper left",
            fontsize=9,
            title_fontsize=10,
            frameon=True,
        )

    # Subplot Formatting
    ax.set_title(title, fontsize=13, pad=8, weight="bold")
    ax.set_xlabel("")
    if ax is not ax1:
        ax.set_ylim(0, 20)
        ax.tick_params(axis="x", rotation=0)
    else:
        ax.tick_params(axis="x", rotation=90)
    ax.set_ylabel("Peak Diff (deg)", fontsize=11)

    ax.set_ylim(ylim)
    ax.grid(True, axis="y", linestyle=":", alpha=0.6)

# Label X-axis on the bottom subplot only
ax4.set_xlabel("Trial Number", fontsize=12, labelpad=10)

plt.tight_layout()
plt.savefig("all_trials_full_column_ticks.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(16, 6))
# Draw light boxplots for each trial column tick
sns.boxplot(
    data=summary_df,
    x="Trial",
    y="Peak_Difference",
    hue="trial_legend",  # Category mapping for legend
    palette=color_palette,  # Colors pulled directly from summary_df['color']
    boxprops=dict(alpha=0.4),
    showfliers=False,
)

# Overlay actual peak points with jitter so every data point is visible
sns.stripplot(
    data=summary_df,
    x="Trial",
    y="Peak_Difference",
    hue="trial_legend",
    palette=color_palette,
    dodge=False,
    size=5,
    jitter=0.2,
    alpha=0.8,
    legend=False,  # Suppresses duplicate legend entries
)

# Position and format the legend
plt.legend(title="Trial Detail", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)

plt.xticks(rotation=45, ha="right")
plt.xlabel("Trial ID (Column Ticks)", fontsize=12, labelpad=10)
plt.ylabel("Joint Displacement (degrees)", fontsize=12)
plt.title("Joint Displacement per Trial", fontsize=14, pad=15)
plt.grid(True, axis="y", linestyle=":", alpha=0.6)

In [ ]:
output_folder = Path("trial_protractor_plots")
output_folder.mkdir(parents=True, exist_ok=True)

records = []

# Sort records by trial_num, then formatted_dur_num
sorted_files = sorted(
    FILES_PROTRACTOR_DIR,
    key=lambda file: (
        parse_file_trial_duration_info(file)["trial_num"],
        parse_file_trial_duration_info(file)["trial_label"],
        parse_file_trial_duration_info(file)["formatted_dur_num"],
    ),
)

for file in sorted_files:
    info = parse_file_trial_duration_info(file)
    trial_num = info["trial_num"]
    trial_label = info["trial_label"]
    formatted_dur_num = info["formatted_dur_num"]

    df = pd.read_csv(file)

    df_filtered = filter_dataframe(df)

    # Find angle difference
    angle_signal = df_filtered["angle"].dropna().values
    peaks, properties = find_peaks(angle_signal, prominence=1)
    prominences = properties["prominences"]

    for p in prominences:
        if trial_num is None:
            continue

        records.append(
            {
                "Trial_Num": trial_num,
                "Trial": trial_label,
                "Peak_Difference": p,
                "color": trial_protractor_map[str(formatted_dur_num)]["color"],
                "trial_legend": trial_protractor_map[str(formatted_dur_num)]["trial_detail"],
            }
        )

    # Plot angle over time
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(28, 16), layout="constrained", gridspec_kw={"height_ratios": [1, 2, 1, 1]})

    # Title for entire figure
    fig.suptitle(
        f"Trial {trial_num} - {trial_protractor_map[str(formatted_dur_num)]['trial_detail'] if trial_num is not None else 'Test'}",
        fontsize=16,
        fontweight="bold",
    )

    # Convert frames into seconds
    df_filtered["time"] = pd.to_numeric(df_filtered["frame"], errors="coerce") / 60  # Assuming 60 FPS

    # Coordinates over time
    ax1.plot(df_filtered["time"], df_filtered["TC_Middle_Leg_x"], label="TC_Middle_Leg X", color="red", alpha=0.7)
    ax1.plot(df_filtered["time"], df_filtered["TC_Middle_Leg_y"], label="TC_Middle_Leg Y", color="blue", alpha=0.7)
    ax1.plot(df_filtered["time"], df_filtered["FT_Middle_Leg_x"], label="FT_Middle_Leg X", color="darkred", linestyle="--")
    ax1.plot(df_filtered["time"], df_filtered["FT_Middle_Leg_y"], label="FT_Middle_Leg Y", color="darkblue", linestyle="--")

    ax1.set_title("Coordinates of TC and FT Middle Legs Over Time")
    ax1.set_ylabel("Coordinates")
    ax1.set_xlabel("Time (s)")
    ax1.legend()

    # Spatial trajectory of keypoints
    ax2.plot(df_filtered["TC_Front_Leg_x"], df_filtered["TC_Front_Leg_y"], color="blue")
    ax2.plot(df_filtered["TC_Middle_Leg_x"], df_filtered["TC_Middle_Leg_y"], color="green")
    ax2.plot(df_filtered["FT_Middle_Leg_x"], df_filtered["FT_Middle_Leg_y"], color="red")

    ax2.set_title("Spatial Trajectory of Keypoints")
    ax2.set_xlabel("X Coordinate")
    ax2.set_ylabel("Y Coordinate")
    ax2.invert_yaxis()  # Invert y-axis to match image coordinate system
    ax2.set_aspect("equal", adjustable="box")
    ax2.grid(True, alpha=0.3)

    # Angle over time
    ax3.plot(df_filtered["time"], df_filtered["angle"], color="purple")
    ax3.set_title("Angle Between TCTC and TCFT Vectors Over Time")
    ax3.set_xlabel("Time (s)")
    ax3.set_ylabel("Angle (degrees)")

    ax3.set_xlim(0, 20)  # Set x-axis limit for time from 0 to 20 seconds

    # Angle difference
    # Collect peak differences and labels for each group
    angle_signal = df_filtered["angle"].dropna().values
    peaks, properties = find_peaks(angle_signal, prominence=1)
    peak_prominences = properties["prominences"]

    if len(peak_prominences) > 0:
        # horizontal Violin Plot (vert=False) placed at y = 1
        parts = ax4.violinplot([peak_prominences], positions=[1], vert=False, showmeans=True, showmedians=False)

        # Custom styling
        for pc in parts["bodies"]:
            pc.set_facecolor("purple")
            pc.set_edgecolor("black")
            pc.set_alpha(0.3)

        # add vertical jitter around y = 1 (X stays exact, Y varies slightly)
        y_coords = np.random.normal(loc=1.0, scale=0.04, size=len(peak_prominences))
        ax4.scatter(peak_prominences, y_coords, alpha=0.7, s=50, color="darkblue", edgecolors="w", zorder=3)

    # 4. Axes Formatting
    ax4.set_yticks([1])
    ax4.set_yticklabels([f"{trial_label}"])
    ax4.set_ylim(0.5, 1.5)  # Keeps the trial centered along the Y-axis

    ax4.set_xlabel("Peak Amplitude / Difference (degrees)", fontsize=12)
    ax4.set_ylabel("Trial", fontsize=12)
    ax4.set_title(
        f"Peak Amplitude Distribution - {trial_label} ({trial_protractor_map[str(formatted_dur_num)]['trial_detail'] if trial_num is not None else 'Test'})",
        fontsize=14,
        pad=15,
    )
    ax4.grid(True, axis="x", linestyle=":", alpha=0.6)

    # plt.subplots_adjust(hspace=0.6)
    # Save figure
    fig_path = output_folder / f"plot_wave_{trial_label}.png"
    # fig.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.close(fig)


summary_df = pd.DataFrame(records)
color_palette = dict(zip(summary_df["trial_legend"], summary_df["color"]))


plt.figure(figsize=(16, 6))
# Draw light boxplots for each trial column tick
sns.boxplot(
    data=summary_df,
    x="Trial",
    y="Peak_Difference",
    hue="trial_legend",  # Category mapping for legend
    palette=color_palette,  # Colors pulled directly from summary_df['color']
    boxprops=dict(alpha=0.4),
    showfliers=False,
)

# Overlay actual peak points with jitter so every data point is visible
sns.stripplot(
    data=summary_df,
    x="Trial",
    y="Peak_Difference",
    hue="trial_legend",
    palette=color_palette,
    dodge=False,
    size=5,
    jitter=0.2,
    alpha=0.8,
    legend=False,  # Suppresses duplicate legend entries
)

# Position and format the legend
plt.legend(title="Trial Detail", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)

plt.xticks(rotation=45, ha="right")
plt.xlabel("Trial ID (Column Ticks)", fontsize=12, labelpad=10)
plt.ylabel("Joint Displacement (degrees)", fontsize=12)
plt.title("Joint Displacement per Trial", fontsize=14, pad=15)
plt.grid(True, axis="y", linestyle=":", alpha=0.6)
plt.tight_layout()

In [ ]:
# Plot angle over time
fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    figsize=(18, 12),
    layout="constrained",
    gridspec_kw={"height_ratios": [2, 1]},
)

# Add new column to select the modified trial names
if "Base_Key" not in summary_df.columns:
    summary_df["Base_Key"] = summary_df["Trial_Num"].apply(lambda x: int(x))

subplot_configs = [
    ("1. All Trials", None, None, ax1, (0, 20)),
    ("2. Trials 1 to 12 - No outlier ones", 1, 11, ax2, (0, 20)),
]


for title, min_k, max_k, ax, ylim in subplot_configs:
    # Filter dataframe by range
    if min_k is None and max_k is None:
        sub_df = summary_df[~summary_df["Base_Key"].isin([12, 21])].copy()
    else:
        sub_df = summary_df[(summary_df["Base_Key"] >= min_k) & (summary_df["Base_Key"] <= max_k)]

    if sub_df.empty:
        ax.set_title(f"{title} (No Data Available)", fontsize=12)
        continue

    # Boxplot
    sns.boxplot(
        data=sub_df,
        x="Trial",
        y="Peak_Difference",
        hue="trial_legend",
        palette=color_palette,
        dodge=False,
        boxprops=dict(alpha=0.4),
        showfliers=False,
        legend=False,  # Suppress subplot legend to prevent clutter
        ax=ax,
    )

    # Stripplot
    sns.stripplot(
        data=sub_df,
        x="Trial",
        y="Peak_Difference",
        hue="trial_legend",
        palette=color_palette,
        dodge=False,
        size=5,
        jitter=0.2,
        alpha=0.8,
        legend=False,
        ax=ax,
    )

    if ax != ax1:
        sub_palette = dict(zip(sub_df["trial_legend"], sub_df["color"]))
        sub_patches = [mpatches.Patch(color=c, label=l) for l, c in sub_palette.items()]

        ax.legend(
            handles=sub_patches,
            title="Stimulus Detail",
            bbox_to_anchor=(1.02, 1),  # Places legend just outside the right edge of the subplot
            loc="upper left",
            fontsize=9,
            title_fontsize=10,
            frameon=True,
        )

    # Subplot Formatting
    ax.set_title(title, fontsize=13, pad=8, weight="bold")
    ax.set_xlabel("")
    if ax is not ax1:
        ax.set_ylim(0, 20)
        ax.tick_params(axis="x", rotation=60)
    else:
        ax.tick_params(axis="x", rotation=90)
    ax.set_ylabel("Peak Diff (deg)", fontsize=11)

    ax.set_ylim(ylim)
    ax.grid(True, axis="y", linestyle=":", alpha=0.6)

# Label X-axis on the bottom subplot only
ax4.set_xlabel("Trial Number", fontsize=12, labelpad=10)

plt.tight_layout()
plt.savefig("all_trials_full_column_ticks.png", dpi=300)
plt.show()

In [ ]:
output_folder = Path("trial_retractor_ex11_plots")
output_folder.mkdir(parents=True, exist_ok=True)

records = []

# Sort records by trial_num, then formatted_dur_num
sorted_files = sorted(
    FILES_EX11_DIR,
    key=lambda file: (
        parse_file_trial_duration_info(file)["ind_num"],
        parse_file_trial_duration_info(file)["trial_num"],
        parse_file_trial_duration_info(file)["trial_label"],
        parse_file_trial_duration_info(file)["formatted_dur_num"],
    ),
)

for file in sorted_files:
    info = parse_file_trial_duration_info(file)
    ind_num = info["ind_num"]
    trial_num = info["trial_num"]
    trial_label = info["trial_label"]
    formatted_dur_num = info["formatted_dur_num"]

    df = pd.read_csv(file)

    df_filtered = filter_dataframe(df)

    # Find angle difference
    angle_signal = df_filtered["angle"].dropna().values
    peaks, properties = find_peaks(angle_signal, prominence=1)
    prominences = properties["prominences"]

    for p in prominences:
        if trial_num is None:
            continue

        records.append(
            {
                "Ind_Num": ind_num,
                "Trial_Num": trial_num,
                "Trial": trial_label,
                "Peak_Difference": p,
                "color": trial_protractor_map[str(formatted_dur_num)]["color"],
                "trial_legend": trial_protractor_map[str(formatted_dur_num)]["trial_detail"],
            }
        )

    # Plot angle over time
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(28, 16), layout="constrained", gridspec_kw={"height_ratios": [1, 2, 1, 1]})

    # Title for entire figure
    fig.suptitle(
        f"Ind {ind_num}) Trial {trial_num} - {trial_protractor_map[str(formatted_dur_num)]['trial_detail'] if trial_num is not None else 'Test'}",
        fontsize=16,
        fontweight="bold",
    )

    # Coordinates over time
    ax1.plot(df_filtered["time"], df_filtered["TC_Middle_Leg_x"], label="TC_Middle_Leg X", color="red", alpha=0.7)
    ax1.plot(df_filtered["time"], df_filtered["TC_Middle_Leg_y"], label="TC_Middle_Leg Y", color="blue", alpha=0.7)
    ax1.plot(df_filtered["time"], df_filtered["FT_Middle_Leg_x"], label="FT_Middle_Leg X", color="darkred", linestyle="--")
    ax1.plot(df_filtered["time"], df_filtered["FT_Middle_Leg_y"], label="FT_Middle_Leg Y", color="darkblue", linestyle="--")

    ax1.set_title("Coordinates of TC and FT Middle Legs Over Time")
    ax1.set_ylabel("Coordinates")
    ax1.set_xlabel("Time (s)")
    ax1.legend()

    # Spatial trajectory of keypoints
    ax2.plot(df_filtered["TC_Front_Leg_x"], df_filtered["TC_Front_Leg_y"], color="blue")
    ax2.plot(df_filtered["TC_Middle_Leg_x"], df_filtered["TC_Middle_Leg_y"], color="green")
    ax2.plot(df_filtered["FT_Middle_Leg_x"], df_filtered["FT_Middle_Leg_y"], color="red")

    ax2.set_title("Spatial Trajectory of Keypoints")
    ax2.set_xlabel("X Coordinate")
    ax2.set_ylabel("Y Coordinate")
    ax2.invert_yaxis()  # Invert y-axis to match image coordinate system
    ax2.set_aspect("equal", adjustable="box")
    ax2.grid(True, alpha=0.3)

    # Angle over time
    ax3.plot(df_filtered["time"], df_filtered["angle"], color="purple")
    ax3.set_title("Angle Between TCTC and TCFT Vectors Over Time")
    ax3.set_xlabel("Time (s)")
    ax3.set_ylabel("Angle (degrees)")

    ax3.set_xlim(0, 20)  # Set x-axis limit for time from 0 to 20 seconds

    # Angle difference
    # Collect peak differences and labels for each group
    angle_signal = df_filtered["angle"].dropna().values
    peaks, properties = find_peaks(angle_signal, prominence=1)
    peak_prominences = properties["prominences"]

    if len(peak_prominences) > 0:
        # horizontal Violin Plot (vert=False) placed at y = 1
        parts = ax4.violinplot([peak_prominences], positions=[1], vert=False, showmeans=True, showmedians=False)

        # Custom styling
        for pc in parts["bodies"]:
            pc.set_facecolor("purple")
            pc.set_edgecolor("black")
            pc.set_alpha(0.3)

        # add vertical jitter around y = 1 (X stays exact, Y varies slightly)
        y_coords = np.random.normal(loc=1.0, scale=0.04, size=len(peak_prominences))
        ax4.scatter(peak_prominences, y_coords, alpha=0.7, s=50, color="darkblue", edgecolors="w", zorder=3)

    # 4. Axes Formatting
    ax4.set_yticks([1])
    ax4.set_yticklabels([f"{trial_label}"])
    ax4.set_ylim(0.5, 1.5)  # Keeps the trial centered along the Y-axis

    ax4.set_xlabel("Peak Amplitude / Difference (degrees)", fontsize=12)
    ax4.set_ylabel("Trial", fontsize=12)
    ax4.set_title(
        f"Peak Amplitude Distribution - {trial_label} ({trial_protractor_map[str(formatted_dur_num)]['trial_detail'] if trial_num is not None else 'Test'})",
        fontsize=14,
        pad=15,
    )
    ax4.grid(True, axis="x", linestyle=":", alpha=0.6)

    # plt.subplots_adjust(hspace=0.6)
    # Save figure
    fig_path = output_folder / f"plot_wave_ind{ind_num}_{trial_label}.png"
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


summary_df = pd.DataFrame(records)
color_palette = dict(zip(summary_df["trial_legend"], summary_df["color"]))

# Plot angle over time
fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    figsize=(18, 12),
    layout="constrained",
    gridspec_kw={"height_ratios": [2, 1]},
)

# Add new column to select the modified trial names
if "Base_Key" not in summary_df.columns:
    summary_df["Base_Key"] = summary_df["Trial_Num"].apply(lambda x: int(x))

subplot_configs = [
    ("1. Ind 03 Trials", 3, None, None, ax1, (0, 30)),
    ("2. Ind 04 Trials", 4, None, None, ax2, (0, 40)),
]


for title, ind_num, min_k, max_k, ax, ylim in subplot_configs:
    # Filter dataframe by range
    if min_k is None and max_k is None:
        sub_df = summary_df[summary_df["Ind_Num"] == ind_num].copy()
    else:
        sub_df = summary_df[(summary_df["Base_Key"] >= min_k) & (summary_df["Base_Key"] <= max_k)]

    if sub_df.empty:
        ax.set_title(f"{title} (No Data Available)", fontsize=12)
        continue

    # Boxplot
    sns.boxplot(
        data=sub_df,
        x="Trial",
        y="Peak_Difference",
        hue="trial_legend",
        palette=color_palette,
        dodge=False,
        boxprops=dict(alpha=0.4),
        showfliers=False,
        legend=False,  # Suppress subplot legend to prevent clutter
        ax=ax,
    )

    # Stripplot
    sns.stripplot(
        data=sub_df,
        x="Trial",
        y="Peak_Difference",
        hue="trial_legend",
        palette=color_palette,
        dodge=False,
        size=5,
        jitter=0.2,
        alpha=0.8,
        legend=False,
        ax=ax,
    )

    if ax != ax1:
        sub_palette = dict(zip(sub_df["trial_legend"], sub_df["color"]))
        sub_patches = [mpatches.Patch(color=c, label=l) for l, c in sub_palette.items()]

        ax.legend(
            handles=sub_patches,
            title="Stimulus Detail",
            bbox_to_anchor=(1.02, 1),  # Places legend just outside the right edge of the subplot
            loc="upper left",
            fontsize=9,
            title_fontsize=10,
            frameon=True,
        )

    # Subplot Formatting
    ax.set_title(title, fontsize=13, pad=8, weight="bold")
    ax.set_xlabel("")
    if ax is not ax1:
        ax.set_ylim(0, 20)
        ax.tick_params(axis="x", rotation=60)
    else:
        ax.tick_params(axis="x", rotation=90)
    ax.set_ylabel("Peak Diff (deg)", fontsize=11)

    ax.set_ylim(ylim)
    ax.grid(True, axis="y", linestyle=":", alpha=0.6)

# Label X-axis on the bottom subplot only
ax4.set_xlabel("Trial Number", fontsize=12, labelpad=10)

plt.tight_layout()
plt.savefig("all_trials_full_column_ticks.png", dpi=300)
plt.show()

In [ ]:
# Plotting angle over time for each individual and plots all the trials in a ascending order of the duration time.
for files in (FILES_PROTRACTOR_DIR, experiment_11_files_ind03, experiment_11_files_ind04):
    sorted_files = sorted(
        files, key=lambda file: (parse_file_trial_duration_info(file)["ind_num"], parse_file_trial_duration_info(file)["formatted_dur_num"])
    )

    unique_durations = sorted(set(parse_file_trial_duration_info(file)["formatted_dur_num"] for file in sorted_files))
    num_durations = len(unique_durations)
    processed_data = []
    ind_num = None  # Initialize ind_num outside the loop

    for file in sorted_files:
        info = parse_file_trial_duration_info(file)
        ind_num = info["ind_num"]
        trial_num = info["trial_num"]
        trial_label = info["trial_label"]
        formatted_dur_num = info["formatted_dur_num"]

        df = pd.read_csv(file)

        df_filtered = filter_dataframe(df)

        angle_signal = df_filtered["angle"].dropna().values
        peaks, peak_properties = find_peaks(angle_signal, prominence=1)
        peak_prominences = peak_properties["prominences"]

        processed_data.append((trial_num, df_filtered, peaks, peak_properties, formatted_dur_num))

    # Plot angle over time
    fig, axs = plt.subplots(num_durations, 1, figsize=(1 * num_durations, 4 * num_durations), layout="constrained")

    if num_durations == 1:
        axs = [axs]
    duration_to_ax_map = {dur: ax for dur, ax in zip(unique_durations, axs)}

    for trial_num, df_filtered, peaks, peak_properties, formatted_dur_num in processed_data:
        # Get the specific subplot axis assigned to this duration
        ax = duration_to_ax_map[formatted_dur_num]

        # Plot the current trial onto the designated duration subplot (overlapping automatically)
        ax.plot(
            df_filtered["time"],
            df_filtered["angle_from_initial"],
            label=f"Trial {trial_num}",
            # color=trial_protractor_map[str(formatted_dur_num)]['color'],
            alpha=0.7,
        )
        # ax.plot(df_filtered['time'].iloc[peaks], df_filtered['angle'].iloc[peaks], markersize=5)
        ax.vlines(
            df_filtered["time"].iloc[peaks],
            df_filtered["angle_from_initial"].iloc[peaks] - peak_properties["prominences"],
            df_filtered["angle_from_initial"].iloc[peaks],
            color="red",
            alpha=0.3,
        )
        ax.legend(loc="upper right", fontsize=8)
        if not ax.get_title():
            detail = trial_protractor_map[str(formatted_dur_num)]["trial_detail"] if trial_num is not None else "Test"
            ax.set_title(f"Duration: {detail}", fontsize=12)
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Angle (degrees)")
            ax.grid(True, alpha=0.3)

    fig.suptitle(f"Individual {ind_num} - Angle Changes Across All Durations", fontsize=16, fontweight="bold")
    plt.savefig(f"ind{ind_num}_all_trials_for_each_duration.png", dpi=300)
    plt.show()

In [ ]:
# Plotting angle over time for each individual and plots all the trials in a ascending order of the duration time.
for files in (experiment_12_files_ind05, experiment_12_files_ind06):
    sorted_files = sorted(
        files,
        key=lambda file: (
            parse_file_trial_duration_info(file)["ind_num"],
            parse_file_trial_duration_info(file)["freq_num"],
            parse_file_trial_duration_info(file)["pulse_width_num"],
        ),
    )
    print(files)

    unique_freq = sorted(set(parse_file_trial_duration_info(file)["freq_num"] for file in sorted_files))
    num_freq = len(unique_freq)
    processed_data = []
    ind_num = None  # Initialize ind_num outside the loop

    for file in sorted_files:
        info = parse_file_trial_duration_info(file)
        ind_num = info["ind_num"]
        trial_num = info["trial_num"]
        freq_num = info["freq_num"]
        pulse_width_num = info["pulse_width_num"]
        trial_label = info["trial_label"]
        formatted_dur_num = info["formatted_dur_num"]

        df = pd.read_csv(file)

        df_filtered = filter_dataframe(df)

        angle_signal = df_filtered["angle_from_initial"].dropna().values
        peaks, peak_properties = find_peaks(angle_signal, prominence=1)
        peak_prominences = peak_properties["prominences"]
        peak_heights = angle_signal[peaks]  # Get the actual peak heights from the signal
        processed_data.append((trial_num, df_filtered, peaks, peak_properties, peak_heights, freq_num, pulse_width_num))

    # Plot angle over time
    fig, axs = plt.subplots(num_freq, 1, figsize=(16, 4 * num_freq), layout="constrained")

    if num_freq == 1:
        axs = [axs]
    print(f"Unique frequencies for individual {ind_num}: {unique_freq}")
    freq_to_ax_map = {freq: ax for freq, ax in zip(unique_freq, axs)}

    for trial_num, df_filtered, peaks, peak_properties, peak_heights, freq_num, pulse_width_num in processed_data:
        # Get the specific subplot axis assigned to this frequency
        ax = freq_to_ax_map[freq_num]

        # Plot the current trial onto the designated frequency subplot (overlapping automatically)
        ax.plot(df_filtered["time"], df_filtered["angle_from_initial"], label=f"Trial {trial_num}", alpha=0.7)
        ax.plot(df_filtered["time"].iloc[peaks], df_filtered["angle_from_initial"].iloc[peaks], "ro", markersize=5)
        ax.legend(loc="upper right", fontsize=8)
        if not ax.get_title():
            # detail = trial_protractor_map[str(freq_num)]['trial_detail'] if trial_num is not None else 'Test'
            ax.set_title(f"Frequency: {freq_num}, Pulse Width: {pulse_width_num}", fontsize=12)
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Angle (degrees)")
            ax.grid(True, alpha=0.3)
    plt.savefig(f"ind{ind_num}_all_trials.png", dpi=300)

    plot_data = []
    for trial_data in processed_data:
        trial_position = trial_data[0]
        df_filtered = trial_data[1]
        troughs = trial_data[2]
        # peak_heights = trial_data[4]
        trough_prominences = trial_data[3]["prominences"]

        trough_angles = df_filtered["angle_from_initial"].iloc[troughs]

        if len(trough_angles) > 0:
            for angle in trough_angles:
                plot_data.append(
                    {
                        "Trial": f" {trial_position} ({trial_data[5]}Hz {trial_data[6]}ms)",  # Include frequency and pulse width in the label
                        "Trough Angle": angle,
                        "color": trial_freq_map[str(trial_data[5])]["color"],
                        "trial_legend": trial_freq_map[str(trial_data[5])]["trial_detail"],
                    }
                )

    df = pd.DataFrame(plot_data)
    color_palette = dict(zip(df["trial_legend"], df["color"]))

    plt.figure(figsize=(18, 4))
    sns.boxplot(data=df, x="Trial", y="Trough Angle", width=0.7, hue="trial_legend", palette=color_palette, boxprops=dict(alpha=0.5))
    plt.xticks(rotation=45, ha="right")
    # Position and format the legend
    plt.legend(title="Trial Detail", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)

    fig.suptitle(f"Individual {ind_num} - Angle Changes Across All Frequencies", fontsize=16, fontweight="bold")
    plt.savefig(f"ind{ind_num}_all_trials_for_each_frequency_initial_angle.png", dpi=300)
    plt.show()

In [ ]:
# Plotting angle over time for each individual and plots all the trials in a ascending order of the duration time.
for files in [FILES_EX13_DIR]:
    sorted_files = sorted(
        files,
        key=lambda file: (
            parse_file_trial_duration_info(file)["ind_num"],
            parse_file_trial_duration_info(file)["freq_num"],
            parse_file_trial_duration_info(file)["pulse_width_num"],
        ),
    )
    print(files)

    unique_freq = sorted(set(parse_file_trial_duration_info(file)["freq_num"] for file in sorted_files))
    num_freq = len(unique_freq)
    processed_data = []
    ind_num = None  # Initialize ind_num outside the loop

    for file in sorted_files:
        info = parse_file_trial_duration_info(file)
        ind_num = info["ind_num"]
        trial_num = info["trial_num"]
        freq_num = info["freq_num"]
        pulse_width_num = info["pulse_width_num"]
        trial_label = info["trial_label"]
        formatted_dur_num = info["formatted_dur_num"]

        df = pd.read_csv(file)

        df_filtered = filter_dataframe(df)

        angle_signal = df_filtered["angle_from_initial"].dropna().values
        peaks, peak_properties = find_peaks(angle_signal, prominence=1)
        peak_prominences = peak_properties["prominences"]
        peak_heights = angle_signal[peaks]  # Get the actual peak heights from the signal
        processed_data.append((trial_num, df_filtered, peaks, peak_properties, peak_heights, freq_num, pulse_width_num))

    # Plot angle over time
    fig, axs = plt.subplots(num_freq, 1, figsize=(16, 4 * num_freq), layout="constrained")

    if num_freq == 1:
        axs = [axs]
    print(f"Unique frequencies for individual {ind_num}: {unique_freq}")
    freq_to_ax_map = {freq: ax for freq, ax in zip(unique_freq, axs)}

    for trial_num, df_filtered, peaks, peak_properties, peak_heights, freq_num, pulse_width_num in processed_data:
        # Get the specific subplot axis assigned to this frequency
        ax = freq_to_ax_map[freq_num]

        # Plot the current trial onto the designated frequency subplot (overlapping automatically)
        ax.plot(df_filtered["time"], df_filtered["angle_from_initial"], label=f"Trial {trial_num}", alpha=0.7)
        ax.plot(df_filtered["time"].iloc[peaks], df_filtered["angle_from_initial"].iloc[peaks], "ro", markersize=5)
        ax.legend(loc="upper right", fontsize=8)
        if not ax.get_title():
            # detail = trial_protractor_map[str(freq_num)]['trial_detail'] if trial_num is not None else 'Test'
            ax.set_title(f"Frequency: {freq_num}, Pulse Width: {pulse_width_num}", fontsize=12)
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Angle (degrees)")
            ax.grid(True, alpha=0.3)
    fig_path = f"ind{ind_num}_all_trials.png"
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")

    plot_data = []
    processed_data = sorted(processed_data, key=lambda x: x[0])
    for trial_data in processed_data:
        trial_position = trial_data[0]
        df_filtered = trial_data[1]
        troughs = trial_data[2]
        # peak_heights = trial_data[4]
        trough_prominences = trial_data[3]["prominences"]

        trough_angles = df_filtered["angle_from_initial"].iloc[troughs]

        if len(trough_angles) > 0:
            for angle in trough_angles:
                plot_data.append(
                    {
                        "Trial": f" {trial_position} ({trial_data[5]}Hz {trial_data[6]}ms)",  # Include frequency and pulse width in the label
                        "Trough Angle": angle,
                        "color": trial_freq_map[str(trial_data[5])]["color"],
                        "trial_legend": trial_freq_map[str(trial_data[5])]["trial_detail"],
                    }
                )

    df = pd.DataFrame(plot_data)
    color_palette = dict(zip(df["trial_legend"], df["color"]))

    plt.figure(figsize=(18, 9))
    sns.boxplot(data=df, x="Trial", y="Trough Angle", width=0.7, hue="trial_legend", palette=color_palette, boxprops=dict(alpha=0.5))
    plt.xticks(rotation=45, ha="right")
    # Position and format the legend
    plt.legend(title="Trial Detail", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)

    fig.suptitle(f"Individual {ind_num} - Angle Changes Across All Frequencies", fontsize=16, fontweight="bold")
    plt.savefig(f"ind{ind_num}_all_trials_for_each_frequency_initial_angle.png", dpi=300)
    plt.show()

In [ ]:
# Plotting angle over time for each individual and plots all the trials in a ascending order of the duration time.
for files in [FILES_EX15_DIR]:
    sorted_files = sorted(
        files,
        key=lambda file: (
            parse_file_trial_duration_info(file)["ind_num"],
            parse_file_trial_duration_info(file)["freq_num"],
            parse_file_trial_duration_info(file)["pulse_width_num"],
        ),
    )
    print(files)

    unique_freq = sorted(set(parse_file_trial_duration_info(file)["freq_num"] for file in sorted_files))
    num_freq = len(unique_freq)
    processed_data = []
    ind_num = None  # Initialize ind_num outside the loop

    for file in sorted_files:
        info = parse_file_trial_duration_info(file)
        ind_num = info["ind_num"]
        trial_num = info["trial_num"]
        freq_num = info["freq_num"]
        pulse_width_num = info["pulse_width_num"]
        trial_label = info["trial_label"]
        formatted_dur_num = info["formatted_dur_num"]

        df = pd.read_csv(file)

        df_filtered = filter_dataframe(df, include_likelihood=True)

        # Pre-processing data for stimulation ON
        # df_filtered = df_filtered.reset_index(drop=True)
        condition = df_filtered["PWM_Light_likelihood"].between(0.8, 1.0)  # If probability is higher than 0.8 count as light is on
        df_filtered["PWM_Time_Period"] = (condition != condition.shift()).cumsum()  # create group ID to change based on periods

        # Filter for the condition and get the start/end time series for each periodic block
        spans = (
            df_filtered[condition]
            .groupby("PWM_Time_Period")
            .agg(start_frame=("frame", "min"), end_frame=("frame", "max"), start_time=("time", "min"), end_time=("time", "max"))
            .reset_index(drop=True)
        )

        angle_signal = df_filtered["angle_from_initial"].dropna().values
        peaks, peak_properties = find_peaks(angle_signal, prominence=1)
        peak_prominences = peak_properties["prominences"]
        peak_heights = angle_signal[peaks]  # Get the actual peak heights from the signal

        troughs, troughs_properties = find_peaks(-angle_signal, width=7, prominence=1)
        processed_data.append((trial_num, df_filtered, troughs, troughs_properties, peak_heights, freq_num, pulse_width_num, spans))

    # Plot angle over time
    fig, axs = plt.subplots(num_freq, 1, figsize=(16, 4 * num_freq), layout="constrained")

    if num_freq == 1:
        axs = [axs]
    print(f"Unique frequencies for individual {ind_num}: {unique_freq}")
    freq_to_ax_map = {freq: ax for freq, ax in zip(unique_freq, axs)}

    for trial_num, df_filtered, troughs, troughs_properties, peak_heights, freq_num, pulse_width_num, spans in processed_data:
        # Get the specific subplot axis assigned to this frequency
        ax = freq_to_ax_map[freq_num]

        # filter the first times without stimulation
        start_time = spans["start_time"].iloc[0]
        filtered_timeseries_data = df_filtered[df_filtered["time"] >= start_time]
        shifted_time = filtered_timeseries_data["time"] - start_time

        # Plot the current trial onto the designated frequency subplot (overlapping automatically)
        ax.plot(shifted_time, filtered_timeseries_data["angle_from_initial"], label=f"Trial {trial_num}", alpha=0.7)
        ax.plot(df_filtered["time"].iloc[troughs] - start_time, df_filtered["angle_from_initial"].iloc[troughs], "ro", markersize=5)
        ax.vlines(
            x=df_filtered["time"].iloc[troughs] - start_time,
            ymin=0,
            ymax=df_filtered["angle_from_initial"].iloc[troughs],
            color="red",
            alpha=0.3,
        )

        # Highlight the time series when electrical stimulation is active (PMW Light ON)
        for i, row in spans.iterrows():
            ax.axvspan(row["start_time"] - start_time, row["end_time"] - start_time, color="blue", alpha=0.1)

        ax.legend(loc="upper right", fontsize=8)

        if not ax.get_title():
            # detail = trial_protractor_map[str(freq_num)]['trial_detail'] if trial_num is not None else 'Test'
            ax.set_title(f"Frequency: {freq_num}, Pulse Width: {pulse_width_num}", fontsize=12)
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Angle (degrees)")
            ax.grid(True, alpha=0.3)

    plt.savefig(f"ind{ind_num}_all_trials.png", dpi=300)

    # Processing data for the boxpllot, peak height quantification
    plot_data = []
    for trial_data in processed_data:
        trial_position = trial_data[0]
        df_filtered = trial_data[1]
        troughs = trial_data[2]
        # peak_heights = trial_data[4]
        trough_prominences = trial_data[3]["prominences"]

        trough_angles = df_filtered["angle_from_initial"].iloc[troughs]

        if len(trough_angles) > 0:
            for angle in trough_angles:
                plot_data.append(
                    {
                        "Trial": f" {trial_position} ({trial_data[5]}Hz {trial_data[6]}ms)",  # Include frequency and pulse width in the label
                        "Trough Angle": angle,
                        "color": trial_freq_map[str(trial_data[5])]["color"],
                        "trial_legend": trial_freq_map[str(trial_data[5])]["trial_detail"],
                    }
                )

    df = pd.DataFrame(plot_data)
    color_palette = dict(zip(df["trial_legend"], df["color"]))

    plt.figure(figsize=(18, 4))
    sns.boxplot(data=df, x="Trial", y="Trough Angle", width=0.7, hue="trial_legend", palette=color_palette, boxprops=dict(alpha=0.5))
    plt.xticks(rotation=45, ha="right")
    # Position and format the legend
    plt.legend(title="Trial Detail", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)

    fig.suptitle(f"Individual {ind_num} - Angle Changes Across All Frequencies", fontsize=16, fontweight="bold")
    plt.savefig(f"ind{ind_num}_all_trials_for_each_frequency_initial_angle.png", dpi=300)
    plt.show()


In [ ]:
# Show only once period of plot for all the trials in a single plot,
# to compare the angle changes across different frequencies.
fig, ax = plt.subplots()
for trial_data in processed_data:
    df_filtered = trial_data[1]
    spans = trial_data[7]
    freq_num = trial_data[5]
    first_period = spans["start_time"].iloc[1]
    start_time = spans["start_time"].iloc[0]
    filtered_timeseries_data = df_filtered[(df_filtered["time"] >= start_time) & (df_filtered["time"] <= first_period)]
    shifted_time = filtered_timeseries_data["time"] - start_time

    mapped_color = trial_freq_map[str(freq_num)]["color"]

    # Plot the current trial onto the designated frequency subplot (overlapping automatically)
    # if freq_num in (10,20,30,40,50):
    ax.plot(shifted_time, filtered_timeseries_data["angle_from_initial"], label=f"{freq_num} Hz", color=mapped_color, alpha=0.7)

handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), loc="upper right", fontsize=8)
ax.set_ylabel("Angle (degrees)")
ax.set_xlabel("Time (s)")
ax.set_ylim(-12.5, 1.5)
ax.set_title("Joint Displacement for First Stimulation Period")
plt.savefig(f"ind{ind_num}_freq_all.png", dpi=300)